# 5교시 · 데이터 시각화
### — 남에게 보여 주기

앞 시간까지는 **내가 보려고** 숫자를 냈습니다.
이번 시간에는 **남이 보게** 만듭니다. 여기서부터는 성격이 달라집니다.

숫자는 틀리지 않았는데 그림 때문에 **상대가 잘못 이해하는 일**이 자주 생깁니다.
이번 시간에는 그림을 그리는 방법과 함께, **그림이 사람을 오해하게 만드는 방식**도 같이 봅니다.

**이 시간이 끝나면 할 수 있는 것**

1. 말하려는 내용에 맞는 차트 종류를 고를 수 있다
2. Matplotlib 으로 제목·축이름·범례를 직접 붙일 수 있다
3. Seaborn 으로 분포와 관계를 짧은 코드로 그릴 수 있다
4. **같은 데이터로 정반대 인상을 주는 그래프를 만들 수 있고, 그래서 조심할 수 있다**


## 준비 — 데이터와 한글 폰트

이 노트북은 **혼자서도 처음부터 끝까지 실행되도록** 만들어져 있습니다.
4교시에서 이미 했더라도, 아래 세 셀을 **다시 한 번 실행**합니다.
Colab 은 노트북을 새로 열 때마다 폰트 설치가 초기화되기 때문입니다.

### 오늘 쓰는 데이터 — 문구·가구 유통사 주문 내역

| | |
|---|---|
| **무엇** | 어느 문구·가구 유통사의 주문 내역 (Tableau 공식 샘플 데이터) |
| **기간** | 2023-01-03 ~ 2026-12-30 (4년치) |
| **크기** | 10,239행 × 21열 · 주문 5,111건 · 고객 804명 |
| **한 행은** | 주문이 아니라 **주문에 담긴 품목 하나**입니다 |
| **지역** | 미국(10,038) · 캐나다(201) |

**주요 열**

| 열 | 뜻 |
|---|---|
| `Order ID` · `Order Date` · `Ship Date` | 주문번호 · 주문일 · 배송일 |
| `Customer ID` · `Segment` | 고객 · 고객 유형(Consumer / Corporate / Home Office) |
| `Region` · `State/Province` · `City` | 지역(Central / East / South / West) · 주 · 도시 |
| `Category` · `Sub-Category` · `Product Name` | 대분류(3종) · 소분류(17종) · 제품명 |
| `Sales` · `Quantity` · `Discount` · `Profit` | 매출 · 수량 · 할인율 · 이익 |

> **결측치·이상치·중복값이 일부러 들어 있습니다.**
> 실무에서 받는 데이터가 그렇기 때문입니다. 손대지 않은 원본이 필요하면
> `superstore_orders_raw.csv` 를 쓰세요.

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders = pd.read_csv(BASE + 'superstore_orders.csv', parse_dates=['Order Date', 'Ship Date'])
orders = orders.drop_duplicates()

print(orders.shape)

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)      # 마이너스 기호도 깨지므로 함께 설정

plt.plot([1, 2, 3], [1, 4, 2])
plt.title('한글이 보이면 성공입니다')
plt.show()

---
# 5-1. 차트는 무엇을 위해 그리는가

먼저 표를 하나 만들어 보겠습니다. 지역별 매출입니다.

In [ ]:
by_region = orders.groupby('Region')['Sales'].sum().sort_values(ascending=False)

by_region.round(0)

## 이 표를 그대로 보고해도 됩니다

숫자 네 개뿐입니다. 표로 봐도 어렵지 않습니다.
그런데 회의에서 이 표를 띄우면 사람들은 이렇게 합니다.

1. 801,749 를 읽는다
2. 739,516 을 읽는다
3. **머릿속에서 두 숫자를 비교한다**
4. 나머지 두 개도 같은 일을 반복한다

**표는 읽는 사람에게 계산을 시킵니다.** 그림은 그 계산을 대신 해 줍니다.

In [ ]:
by_region.plot(kind='bar', figsize=(7, 4))
plt.title('지역별 매출')
plt.show()

막대 길이를 보는 순간 순서와 차이가 **한 번에** 들어옵니다.
비교하는 일을 그림이 대신 해 줬기 때문입니다.

> ### 여기서 기억할 것
> 차트는 **예쁘게 만들려고** 그리는 것이 아닙니다.
> **읽는 사람이 머릿속에서 해야 할 비교를, 그림이 대신 해 주려고** 그립니다.
>
> 그래서 순서가 이렇게 됩니다.
> **① 내가 무엇을 말하려는지 정한다 → ② 그 말에 맞는 차트를 고른다 → ③ 그린다**
>
> 반대로 하면 안 됩니다. 일단 그려 놓고 "여기서 뭐가 보이지?" 하고 찾는 것은
> 나 혼자 탐색할 때는 괜찮지만, **남에게 보여 줄 차트를 만드는 방법은 아닙니다.**

| | 표 | 그림 |
|---|---|---|
| 정확한 값을 알려 준다 | ⭕ | ❌ |
| 크기 차이를 한눈에 보여 준다 | ❌ | ⭕ |
| 항목이 20개 넘어도 읽을 수 있다 | ⭕ | ❌ |
| 추세·패턴을 보여 준다 | ❌ | ⭕ |

**둘 중 하나가 더 좋은 게 아닙니다.** 정확한 숫자가 필요하면 표,
크기 비교나 흐름을 보여 주려면 그림입니다. 자료에 둘 다 넣어도 됩니다.

---
# 5-2. 무엇을 말하려는가에 따라 차트가 정해진다

차트 종류는 수십 가지가 있지만, **업무에서 쓰는 것은 네 가지**입니다.
그리고 그 넷은 **말하려는 내용**으로 구분됩니다.

| 말하려는 것 | 예시 문장 | 차트 | 코드 |
|---|---|---|---|
| **비교** | "동부가 남부보다 두 배 판다" | 막대그래프 | `kind='bar'` |
| **추이** | "매출이 하반기에 올라간다" | 선그래프 | `kind='line'` |
| **관계** | "할인을 많이 하면 이익이 준다" | 산점도 | `kind='scatter'` |
| **분포** | "주문 대부분은 소액이다" | 히스토그램 | `kind='hist'` |

네 가지를 실제로 하나씩 그려 보겠습니다.

## ① 비교 — 막대그래프

**항목끼리 크기를 견주어 볼 때** 씁니다.
막대는 **길이**로 크기를 나타내기 때문에 사람 눈이 가장 정확하게 비교합니다.

In [ ]:
by_subcat = orders.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False).head(10)

by_subcat.plot(kind='barh', figsize=(8, 5))
plt.title('매출 상위 10개 품목')
plt.xlabel('매출')
plt.show()

항목 이름이 길면 **`kind='barh'` (가로 막대)** 를 쓰세요.
세로 막대(`kind='bar'`)로 그리면 글자가 비스듬히 누워서 읽기 어렵습니다.

그리고 **정렬**을 했습니다. `sort_values(ascending=False)` 한 줄입니다.
정렬하지 않은 막대그래프는 순위를 읽으려면 눈이 왔다 갔다 해야 합니다.

## ② 추이 — 선그래프

**시간에 따라 어떻게 변했는지** 보여 줄 때 씁니다.
점을 선으로 이어 놓기 때문에 **"이어져 있다"** 는 느낌을 줍니다.
그래서 **가로축이 시간일 때만** 쓰는 것이 원칙입니다.

In [ ]:
orders['year_month'] = orders['Order Date'].dt.to_period('M')
monthly_sales = orders.groupby('year_month')['Sales'].sum()
monthly_sales.index = monthly_sales.index.to_timestamp()      # 그림용으로 날짜형으로 되돌립니다

monthly_sales.plot(kind='line', figsize=(10, 4))
plt.title('월별 매출')
plt.ylabel('매출')
plt.show()

위아래로 심하게 흔들립니다. 이 데이터는 **4년치(2023~2026)** 입니다.
전체적으로는 오른쪽으로 올라가지만, 달마다는 들쭉날쭉합니다.

**어느 기간을 잘라서 보여 주느냐에 따라 인상이 달라집니다.**
같은 선인데 7월부터 그리면 급등으로, 4월부터 그리면 급락으로 보입니다.


## ③ 관계 — 산점도

**두 숫자가 서로 관련이 있는지** 볼 때 씁니다.
점 하나가 데이터 한 행입니다. 여기서는 점 하나가 **주문 한 건**입니다.

In [ ]:
orders.plot(kind='scatter', x='Discount', y='Profit', figsize=(8, 5), alpha=0.3)
plt.title('할인율과 이익')
plt.axhline(0, color='red', linewidth=1)       # 이익 0 기준선
plt.show()

`alpha=0.3` 은 **점을 반투명하게** 만드는 설정입니다.
점이 1만 개나 겹치기 때문에, 이걸 안 하면 까맣게 뭉쳐서 아무것도 안 보입니다.

빨간 선(이익 0) **아래쪽에 있는 점이 적자 주문**입니다.
할인율이 오른쪽으로 갈수록 아래쪽 점이 많아지는 게 보입니다.

숫자로도 확인해 봅시다.

In [ ]:
print('할인율과 이익의 상관계수: {:.3f}'.format(orders['Discount'].corr(orders['Profit'])))
print('적자 주문 비율: {:.1f}%'.format((orders['Profit'] < 0).mean() * 100))

**상관계수 -0.219.** 마이너스니까 "할인이 커질수록 이익은 작아지는 쪽"입니다.
다만 -1 에서 한참 먼 값이라 **약한 관계**입니다. 할인만으로 이익이 정해지지는 않습니다.

> 상관계수는 -1 ~ +1 사이 값입니다. 0 에 가까우면 관계가 거의 없고,
> ±1 에 가까우면 한쪽이 변할 때 다른 쪽도 규칙적으로 변합니다.
>
> **그리고 관계가 있다는 것이 원인이라는 뜻은 아닙니다.**
> 할인을 많이 해서 이익이 준 것인지, 원래 안 팔리는 물건이라 할인을 많이 한 것인지
> 이 숫자만으로는 알 수 없습니다. **상관관계는 인과관계가 아닙니다.**


In [ ]:
# ④ 분포 — 히스토그램
orders[orders['Sales'] < 1000]['Sales'].plot(kind='hist', bins=50, figsize=(9, 4))
plt.title('주문 금액 분포 (1,000 미만)')
plt.xlabel('주문 금액')
plt.show()

> ### 여기서 기억할 것
> **차트 종류를 고르는 일은 취향이 아닙니다.** 말하려는 내용이 정해 줍니다.
>
> - 항목 비교 → **막대**
> - 시간 흐름 → **선**
> - 두 숫자의 관계 → **산점도**
> - 값이 퍼진 모양 → **히스토그램**
>
> 하나만 더 — **원그래프(파이차트)는 되도록 쓰지 마세요.**
> 사람은 **각도**보다 **길이**를 훨씬 정확하게 비교합니다.
> 조각이 3개를 넘어가면 어느 쪽이 큰지 눈으로 판단이 안 됩니다. 그럴 땐 막대가 낫습니다.

---
# 5-3. Matplotlib 기본 — 직접 그리기

지금까지는 `df.plot(...)` 을 썼습니다. **판다스가 대신 그려 준 것**입니다.
편하지만, 세밀하게 손보려면 **Matplotlib 을 직접** 쓰는 편이 낫습니다.

`df.plot()` 도 사실은 내부에서 Matplotlib 을 부릅니다.
**같은 도구인데, 판다스를 거치느냐 직접 부르느냐의 차이**입니다.

In [ ]:
plt.figure(figsize=(8, 5))                        # ① 그림판 크기를 먼저 정합니다

plt.bar(by_region.index, by_region.values, color='#4C72B0')   # ② 막대를 그립니다

plt.title('동부 지역이 전체 매출의 32.9%를 차지합니다')   # ③ 제목
plt.xlabel('지역')                                 # ④ 가로축 이름
plt.ylabel('매출')                                 # ⑤ 세로축 이름

plt.show()                                        # ⑥ 화면에 띄웁니다

## 한 줄씩 무슨 뜻인지

| 코드 | 하는 일 |
|---|---|
| `plt.figure(figsize=(8, 5))` | **그림판을 준비합니다.** 가로 8, 세로 5 (인치 단위) |
| `plt.bar(x, y)` | 막대를 그립니다. 선은 `plt.plot`, 점은 `plt.scatter` |
| `color='#4C72B0'` | 색을 지정합니다. `'red'` 처럼 이름으로도, `'#4C72B0'` 처럼 코드로도 됩니다 |
| `plt.title('...')` | 제목을 붙입니다 |
| `plt.xlabel` / `plt.ylabel` | 가로축·세로축 이름을 붙입니다 |
| `plt.show()` | **여기까지 그린 것을 화면에 띄웁니다.** 이걸 부르면 그림판이 비워집니다 |

**`plt.show()` 를 부르기 전까지 명령이 계속 같은 그림에 쌓입니다.**
그래서 `plt.bar` 로 막대를 그리고, 그 뒤에 `plt.title` 로 제목을 얹는 게 가능합니다.

`plt.show()` 를 안 쓰면 다음 셀의 그림과 겹쳐 그려질 수 있습니다. **항상 마지막에 넣으세요.**

## 범례 — 선이 여러 개일 때

한 그림에 선을 여러 개 그리면 **어느 선이 뭔지** 알려 줘야 합니다.
그게 **범례(legend)** 입니다.

`label=` 로 이름을 붙이고 `plt.legend()` 를 부르면 됩니다.

In [ ]:
cat_monthly = orders.pivot_table(index='year_month', columns='Category',
                                values='Sales', aggfunc='sum')
cat_monthly.index = cat_monthly.index.to_timestamp()

plt.figure(figsize=(11, 4))

for name in ['Furniture', 'Office Supplies', 'Technology']:
    plt.plot(cat_monthly.index, cat_monthly[name], label=name)

plt.title('카테고리별 월 매출 추이')
plt.ylabel('매출')
plt.legend()                    # 이 한 줄이 범례를 만듭니다
plt.grid(alpha=0.3)             # 옅은 격자선 (값을 읽기 쉬워집니다)
plt.show()

---
# 5-4. Seaborn — 짧은 코드로 분포와 관계

**Seaborn(시본)** 은 Matplotlib 위에 얹혀 있는 도구입니다.
Matplotlib 으로 열 줄 걸리는 것을 한 줄로 그려 줍니다. **Colab 에는 이미 설치돼 있습니다.**

Seaborn 함수들은 대부분 이 모양입니다.

```python
sns.무슨그림(data=표, x='가로축열', y='세로축열')
```

**표를 통째로 넘기고, 열 이름만 알려 주면 됩니다.**

In [ ]:
import seaborn as sns

sns.set_theme(style='whitegrid', font='NanumGothic')   # 보기 좋은 기본 설정 + 한글 폰트
plt.rc('axes', unicode_minus=False)

print(sns.__version__)

In [ ]:
# ① 분포 — `sns.histplot`
plt.figure(figsize=(9, 4))

sns.histplot(data=orders[orders['Sales'] < 1000], x='Sales', bins=50)

plt.title('주문의 절반이 53.7 이하입니다')
plt.xlabel('주문 금액')
plt.ylabel('주문 건수')
plt.show()

In [ ]:
# ③ 관계 — `sns.scatterplot`
plt.figure(figsize=(9, 5))

sns.scatterplot(data=orders, x='Discount', y='Profit',
                hue='Category', alpha=0.4)

plt.title('할인율이 높아질수록 적자 주문이 늘어납니다')
plt.axhline(0, color='red', linewidth=1)
plt.show()

In [ ]:
# ④ 비교 — sns.barplot (기본은 평균이므로 estimator="sum" 으로 합계 지정)
plt.figure(figsize=(8, 4))

sns.barplot(data=orders, x='Region', y='Sales',
            estimator='sum', errorbar=None,
            order=['East', 'West', 'Central', 'South'])   # 큰 순서로 직접 지정

plt.title('지역별 매출 합계')
plt.ylabel('매출 합계')
plt.show()

In [ ]:
# ⑤ 상자그림 — `sns.boxplot`
plt.figure(figsize=(9, 5))

sns.boxplot(data=orders[orders['Sales'] < 1000], x='Category', y='Sales')

plt.title('카테고리별 주문 금액 (1,000 미만)')
plt.ylabel('주문 금액')
plt.show()

## Matplotlib 과 Seaborn, 언제 뭘 쓰나

| | Matplotlib | Seaborn |
|---|---|---|
| 코드 길이 | 길다 | 짧다 |
| 그룹별로 나눠 그리기 | 반복문 필요 | `hue=` 한 줄 |
| 세밀하게 손보기 | 자유롭다 | 제한이 있다 |
| 기본 모양 | 밋밋하다 | 보기 좋다 |

**둘 중 하나를 고르는 게 아닙니다.**
Seaborn 으로 그려 놓고 `plt.title` · `plt.ylabel` 로 다듬는 것이 실제로 가장 흔한 방식입니다.
위 코드들도 전부 그렇게 했습니다.

---
# 5-6. 제목을 결론 문장으로 쓰기

이 절은 코드가 거의 없습니다. 그런데 **효과는 오늘 배우는 것 중 가장 큽니다.**

대부분의 사람이 차트 제목을 이렇게 씁니다.

- ❌ 지역별 매출
- ❌ 월별 추이
- ❌ 할인율과 이익의 관계

이건 **제목이 아니라 이름표**입니다. 그림을 보면 이미 아는 내용입니다.
**아무 정보도 더해 주지 않습니다.**

제목 자리에는 **당신이 이 그림을 보고 내린 결론**을 쓰세요.

- ⭕ 동부 지역이 전체 매출의 32.9%를 차지합니다
- ⭕ 매출은 매년 오르지만 1~2월은 항상 저조합니다
- ⭕ 할인율 30%를 넘으면 대부분 적자입니다

**직접 비교해 봅시다.**

In [ ]:
# ❌ 이름표 제목
plt.figure(figsize=(8, 4))
plt.bar(by_region.index, by_region.values, color='#8C8C8C')
plt.title('지역별 매출')
plt.show()

In [ ]:
# ⭕ 결론 제목 — 말하려는 것을 색으로도 함께 강조합니다
colors = ['#C44E52' if region == 'East' else '#CCCCCC' for region in by_region.index]

plt.figure(figsize=(8, 4))
plt.bar(by_region.index, by_region.values, color=colors)
plt.title('동부 지역 한 곳이 전체 매출의 32.9%를 차지합니다')
plt.ylabel('매출')
plt.show()

## 두 그림의 데이터는 완전히 같습니다

바뀐 것은 **제목 한 줄과 색 하나**뿐입니다. 그런데 전달되는 내용이 다릅니다.

| | 이름표 제목 | 결론 제목 |
|---|---|---|
| 보는 사람이 하는 일 | 그림을 해석한다 | 결론을 확인한다 |
| 해석이 갈릴 가능성 | 있다 | 적다 |
| 만든 사람의 책임 | 애매하다 | **분명하다** |

마지막 줄이 중요합니다.
**결론을 제목에 쓰면, 그 결론이 틀렸을 때 내 책임이 됩니다.**

그래서 부담이 됩니다. "지역별 매출"이라고 써 두면 아무 말도 하지 않은 셈이기 때문입니다.
하지만 **아무 말도 안 하는 자료는 회의 시간만 씁니다.**

> ### 여기서 기억할 것
> **차트 제목은 결론 문장으로 씁니다.**
>
> 결론을 못 쓰겠다면, 그건 제목 문제가 아니라
> **아직 이 차트에서 무엇을 말할지 안 정해졌다는 뜻**입니다.
> 그럴 때는 제목을 고민하지 말고, **그 차트를 왜 그렸는지 다시 생각해 보세요.**

---
# 5-7. 그림은 사람을 오해하게 만들 수 있습니다

**이번 시간의 핵심입니다.**

지금부터 **같은 데이터로 정반대 인상을 주는 그래프를 두 쌍** 만들어 보겠습니다.
미리 말해 둡니다 — **어느 쪽도 데이터를 조작하지 않습니다.**
숫자를 고치지도, 빼지도, 지어내지도 않습니다. **전부 실제 값 그대로입니다.**

## 방법 ① — 세로축을 어디서 시작할 것인가

지역별 매출을 다시 봅니다. 가장 큰 동부가 801,749, 가장 작은 남부가 391,507 입니다.
**남부는 동부의 48.8%** 입니다. 대략 절반쯤 판다는 뜻입니다.

이 "절반쯤"이라는 관계가 그림에서 어떻게 보이는지가 이번 절의 관심사입니다.

먼저 **세로축을 0부터** 그립니다.

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(by_region.index, by_region.values, color='#4C72B0')
plt.ylim(0, 900000)                        # 세로축을 0부터
plt.title('세로축 0부터 — 네 지역의 매출은 비슷한 수준입니다')
plt.ylabel('매출')
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(by_region.index, by_region.values, color='#C44E52')
plt.ylim(380000, 820000)        # 세로축을 380,000부터 잘라냄
plt.title('세로축 잘라냄 — 동부가 남부의 두 배 넘게 팝니다')
plt.ylabel('매출')
plt.show()

## 무슨 일이 일어났습니까

| | 위 그림 (0부터) | 아래 그림 (380,000부터) |
|---|---|---|
| 동부 막대 | 길다 | 길다 |
| 남부 막대 | 조금 짧다 | **거의 없다** |
| 받는 인상 | 네 지역이 비슷하다 | **남부가 완전히 죽었다** |
| 실제 숫자 | 801,749 vs 391,507 | 801,749 vs 391,507 |

**숫자는 한 글자도 안 바뀌었습니다.** 바뀐 건 `plt.ylim` 한 줄입니다.

세로축을 잘라내면 **막대 길이의 비율이 실제 값의 비율과 달라집니다.**
남부 막대는 실제로 동부의 49% 인데, 아래 그림에서는 **길이가 거의 0** 으로 보입니다.

> **막대그래프의 세로축은 0에서 시작하는 것이 원칙입니다.**
> 막대는 **길이로 크기를 말하는** 그림이기 때문입니다. 길이를 자르면 거짓말이 됩니다.
>
> 선그래프는 사정이 다릅니다. 선은 크기가 아니라 **변화의 방향**을 말하기 때문에
> 축을 자르는 게 오히려 나을 때가 있습니다. 다만 그때도 **잘랐다고 밝혀야** 합니다.

## 방법 ③ — 막대 순서를 어떻게 놓을 것인가

마지막입니다. 이것은 조금 더 미묘합니다.

In [ ]:
top5 = orders.groupby('Sub-Category')['Sales'].sum().nlargest(5)

plt.figure(figsize=(8, 4))
plt.bar(top5.index, top5.values, color='#4C72B0')
plt.title('큰 순서로 정렬 — Chairs 가 1위라는 게 바로 보입니다')
plt.ylabel('매출')
plt.show()

In [ ]:
top5_alpha = top5.sort_index()        # 이름 가나다순으로 다시 늘어놓기

plt.figure(figsize=(8, 4))
plt.bar(top5_alpha.index, top5_alpha.values, color='#8C8C8C')
plt.title('이름순 정렬 — 순위가 한눈에 안 들어옵니다')
plt.ylabel('매출')
plt.show()

아래 그림은 **거짓말을 하지 않습니다.** 다만 **정리하지 않은 것**입니다.

값이 들쭉날쭉 늘어서 있으면 보는 사람이 눈으로 순위를 매겨야 합니다.
**"읽는 사람의 비교를 대신해 준다"는 차트의 목적을 스스로 포기한 그림**입니다.

정렬은 사소해 보이지만, **하느냐 안 하느냐로 전달력이 갈립니다.**

## 정리 — 그래서 무엇을 조심해야 합니까

지금까지 만든 그림 중 **어느 것도 데이터를 조작하지 않았습니다.**
그런데 받는 인상은 정반대였습니다. **이게 시각화가 위험한 이유입니다.**

숫자를 고치면 그건 조작이고, 걸리면 큰일이 납니다. 그래서 아무도 안 합니다.
**그런데 축을 자르고 기간을 고르는 건 아무도 잘못이라고 안 합니다.** 효과는 거의 같은데요.

| 조심할 것 | 확인 질문 |
|---|---|
| 세로축 시작점 | 막대그래프인데 0에서 시작하지 않았습니까? |
| 기간 선택 | 왜 하필 그 달부터입니까? 다른 달부터 그리면 결론이 바뀝니까? |
| 정렬 순서 | 순위를 보여 주는 그림인데 정렬을 안 했습니까? |
| 제외한 데이터 | 이상치를 뺐다면 그 사실을 적었습니까? |
| 축 이름·단위 | 단위가 없거나, 두 그림의 축 범위가 다르지 않습니까? |

> ### 여기서 기억할 것
> **차트를 만든 사람은 보는 사람의 이해에 책임이 있습니다.**
>
> "나는 사실만 그렸다"는 변명이 안 되는 이유는,
> **어떤 사실을 어떻게 보여 줄지 고른 사람이 나이기 때문**입니다.
>
> 자기 차트를 남에게 보내기 전에 스스로 한 번 물어보세요.
> **"내가 이 그림을 처음 보는 사람이라면, 실제와 다르게 이해할 여지가 있는가?"**

---
# 5-8. 실습 — 차트를 그리고, 오해를 만들어 보기

**한 셀씩 빈칸을 채우고 실행**하세요.
마지막 두 문제는 **같은 데이터로 정반대 인상**을 만드는 것입니다.

In [ ]:
# ① 고객 유형(`Segment`)별 매출을 막대그래프로 그리세요
seg = orders.groupby('Segment')['Sales'].sum()

seg.plot(kind='bar', figsize=(7, 3))
plt.title('고객 유형별 매출')
plt.show()

In [ ]:
# ② 큰 순서로 정렬해서 다시 그리세요
seg.sort_values(ascending=False).plot(kind='bar', figsize=(7, 3))
plt.title('고객 유형별 매출')
plt.show()

In [ ]:
# ③ 제목을 **결론 문장**으로 바꾸세요 (따옴표 안을 직접 쓰세요)
seg.sort_values(ascending=False).plot(kind='bar', figsize=(7, 3))
plt.title('Consumer 가 세 유형 중 매출이 가장 큽니다')
plt.show()

In [ ]:
# ④ 세로축을 0부터 그리세요 — 정직한 버전
seg.plot(kind='bar', figsize=(7, 3))
plt.ylim(0, seg.max() * 1.1)
plt.title('세로축 0부터')
plt.show()

In [ ]:
# ⑤ 이번엔 세로축을 잘라 보세요 — 차이가 얼마나 커 보입니까
seg.plot(kind='bar', figsize=(7, 3))
plt.ylim(seg.min() * 0.95, seg.max() * 1.02)
plt.title('세로축을 잘라낸 버전')
plt.show()

> ### 실습 3 을 시키는 이유
> **직접 만들어 봐야 남이 만든 걸 알아봅니다.**
>
> 앞으로 회의에서 남의 차트를 볼 때 세로축부터 보게 될 겁니다.
> 그게 이 실습의 목적입니다.

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 판다스로 빠르게 그리기 | `df['열'].plot(kind='bar')` |
| 그림판 크기 정하기 | `plt.figure(figsize=(8, 5))` |
| 막대 / 선 / 점 | `plt.bar(x, y)` · `plt.plot(x, y)` · `plt.scatter(x, y)` |
| 제목 · 축 이름 | `plt.title(...)` · `plt.xlabel(...)` · `plt.ylabel(...)` |
| 범례 | `plt.plot(..., label='이름')` + `plt.legend()` |
| 축 범위 | `plt.ylim(아래, 위)` |
| 기준선 | `plt.axhline(0)` · `plt.axvline(0)` |
| 화면에 띄우기 | `plt.show()` |
| Seaborn 기본 설정 | `sns.set_theme(style='whitegrid', font='NanumGothic')` |
| 분포 | `sns.histplot(data=df, x='열', hue='그룹')` |
| 관계 | `sns.scatterplot(data=df, x='열1', y='열2', hue='그룹')` |
| 비교 | `sns.barplot(data=df, x='그룹', y='값', estimator='sum')` |
| 상자그림 | `sns.boxplot(data=df, x='그룹', y='값')` |
| 한글 폰트 | `!apt-get -qq install fonts-nanum` + `plt.rc('font', family='NanumGothic')` |

## 차트 고르는 표

| 말하려는 것 | 차트 |
|---|---|
| 항목끼리 비교 | 막대 (항목 이름이 길면 가로 막대) |
| 시간에 따른 변화 | 선 |
| 두 숫자의 관계 | 산점도 |
| 값이 퍼진 모양 | 히스토그램 · 상자그림 |

## 남길 것 세 가지

1. **차트 종류는 말하려는 내용이 정한다** — 먼저 문장을 정하고, 그 다음에 차트를 고릅니다
2. **제목에는 이름표가 아니라 결론을 쓴다** — 결론을 못 쓰겠다면 그 차트를 왜 그렸는지 다시 생각합니다
3. **데이터를 조작하지 않아도 오해하게 만들 수 있다** — 축·기간·정렬만으로 인상이 뒤집힙니다

---

### 다음 시간

**남은 시간은 오늘 배운 것을 실제 분석에 써 보는 데 씁니다.**

오늘 그린 그림에서 "동부가 더 많다", "할인이 많으면 이익이 준다" 같은 것을 눈으로 봤습니다.
이제 이 도구들을 하나의 질문에 이어 붙여, **처음부터 끝까지 하나의 분석**을 해 봅니다.
